# EMLProverV2 Explorer
**monogate v1.1.0 — Neurosymbolic Discovery Engine**

This notebook demonstrates the EMLProverV2 exploration loop: an autonomous
conjecture–verify–compress–learn cycle that gets smarter as it runs.

## What you'll see
1. Basic proving with the 4-tier pipeline
2. Temperature-controlled conjecture generation
3. The `explore()` loop in action (5 rounds, ~2 min)
4. Neural scorer learning curve
5. Interactive Plotly proof tree visualization
6. Gallery of auto-discovered identities

## Key insight from experiments
> The neural scorer only learns from identities that reach the MCTS tier (≈14% of the catalog).
> `explore()` is designed to generate exactly those — creative enough to need MCTS,
> but not so wild that they're unprovable.

In [ ]:
# Install (Colab / fresh env)
# !pip install monogate plotly matplotlib -q

import sys, os
# Uncomment if running from repo root:
# sys.path.insert(0, os.path.join(os.getcwd(), 'python'))

In [ ]:
from monogate.prover import EMLProverV2
from monogate.identities import ALL_IDENTITIES, get_by_category, get_by_difficulty
from monogate.neural_scorer import FeatureBasedEMLScorer
import math, json

print(f"monogate loaded — {len(ALL_IDENTITIES)} identities in catalog")

## 1. Basic proving — the 4-tier pipeline

The prover works in tiers:
- **Tier 1:** Numerical (500 probes) — fast, 90% confidence
- **Tier 2:** SymPy exact — slow but certainty = 1.0
- **Tier 3:** Certified interval arithmetic — tight bounds
- **Tier 4:** MCTS witness — finds an EML tree that matches f(x), then verifies symbolically

In [ ]:
prover = EMLProverV2(enable_learning=True)

examples = [
    "sin(x)**2 + cos(x)**2 == 1",          # trivial — Tier 2
    "exp(x) * exp(-x) == 1",               # easy — Tier 2
    "cosh(x)**2 - sinh(x)**2 == 1",        # hyperbolic — Tier 2
    "2*sin(x)*cos(x) == sin(2*x)",         # double angle — Tier 2
]

for expr in examples:
    r = prover.prove(expr)
    symbol = "✓" if r.proved() else "✗"
    print(f"{symbol} [{r.status:20s}] {r.elapsed_s:.3f}s  {expr}")

## 2. Conjecture generation — temperature-controlled mutations

The conjecture engine mutates seed identities from a chosen category.
Temperature controls how aggressively it mutates:

| Temperature | Style | Mutations |
|-------------|-------|-----------|
| 0.2 | Conservative | x→2x, x→x/2 only |
| 0.5 | Moderate | + scaling, negation, shift |
| 0.8 | Creative | + triple-arg, phase shifts, random scale |

In [ ]:
for temp in [0.2, 0.5, 0.8]:
    conjectures = prover.generate_conjectures(
        category="trig", n=5, temperature=temp, seed=42
    )
    print(f"\nTemperature {temp} — {len(conjectures)} conjectures:")
    for c in conjectures:
        print(f"  {c.expression[:70]}")

In [ ]:
# Prove a batch of conjectures and see which tiers they hit
conjectures = prover.generate_conjectures(
    category="trig", n=15, temperature=0.7, seed=99
)
print(f"Generated {len(conjectures)} conjectures — proving...\n")

tier_counts = {}
for c in conjectures:
    r = prover.prove(c.expression)
    s = r.status if r.proved() else "failed"
    tier_counts[s] = tier_counts.get(s, 0) + 1
    print(f"  {'✓' if r.proved() else '✗'} [{s:20s}]  {c.expression[:55]}")

print("\nTier summary:", tier_counts)

## 3. Explorer mode — the full conjecture–verify–learn loop

Each round:
1. Generate creative conjectures from trig seeds
2. Prove each with the 4-tier prover
3. Compress any MCTS witnesses (Occam's razor)
4. The neural scorer learns from MCTS successes automatically

**Why trig?** Our transfer experiments showed trig→all gives 1.05–1.32× speedup
to other domains. It's the best curriculum seed.

In [ ]:
explorer = EMLProverV2(enable_learning=True)

results = explorer.explore(
    n_rounds=5,
    n_per_round=20,
    seed_category="trig",
    temperature=0.7,
    compress_witnesses=True,
    verbose=True,
)

print(f"\nTotal discovered: {results['n_total_discovered']} identities")

## 4. Neural scorer learning curve

The scorer only trains from MCTS witnesses. Track how the buffer fills
as exploration proceeds.

In [ ]:
try:
    import matplotlib.pyplot as plt
    import numpy as np

    curve = results["learning_curve"]
    rounds      = [r["round"] for r in curve]
    n_proved    = [r["n_proved"] for r in curve]
    buf_sizes   = [r["scorer_buffer"] for r in curve]
    n_total     = [r["n_discovered_total"] for r in curve]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].bar(rounds, n_proved, color="steelblue", alpha=0.8)
    axes[0].set_xlabel("Round")
    axes[0].set_ylabel("Proved this round")
    axes[0].set_title("Proofs per round")
    axes[0].set_xticks(rounds)

    axes[1].plot(rounds, buf_sizes, "o-", color="darkorange")
    axes[1].set_xlabel("Round")
    axes[1].set_ylabel("Scorer buffer size")
    axes[1].set_title("Neural scorer data accumulation")
    axes[1].axhline(10, ls="--", color="gray", alpha=0.5, label="min_samples=10")
    axes[1].legend()

    axes[2].plot(rounds, n_total, "s-", color="mediumseagreen")
    axes[2].set_xlabel("Round")
    axes[2].set_ylabel("Total discovered")
    axes[2].set_title("Cumulative discovery")

    fig.suptitle("EMLProverV2 Explorer — Learning Curve", fontsize=14)
    fig.tight_layout()
    plt.show()
    print(f"Scorer trained: {explorer.scorer.is_trained()}  |  Buffer: {len(explorer.scorer._buffer)} samples")
except ImportError:
    print("matplotlib not available — install with: pip install matplotlib")
    for r in results["learning_curve"]:
        print(f"  Round {r['round']}: proved={r['n_proved']}  buffer={r['scorer_buffer']}  total={r['n_discovered_total']}")

## 5. Gallery of discovered identities

In [ ]:
discovered = results["discovered"]

if not discovered:
    print("No identities discovered yet — try more rounds or higher temperature.")
else:
    print(f"Discovered {len(discovered)} identities:\n")
    for i, (identity, proof) in enumerate(discovered[:20], 1):
        tier = proof.status.replace("proved_", "")
        nodes = f" | nodes={proof.node_count}" if proof.node_count > 0 else ""
        print(f"  {i:3d}. [{tier:12s}]{nodes}  {identity.expression[:70]}")

In [ ]:
# Show by proof method
from collections import Counter
methods = Counter(proof.status for _, proof in discovered)
print("Proof methods:")
for method, count in sorted(methods.items(), key=lambda x: -x[1]):
    print(f"  {method:25s}: {count}")

## 6. Interactive proof visualization

When a conjecture is proved via MCTS (Tier 4), the prover finds an
EML witness tree. Visualize it interactively.

In [ ]:
# Find a proved_witness result from exploration
witness_proofs = [
    (identity, proof) for identity, proof in discovered
    if proof.status == "proved_witness" and proof.witness_tree is not None
]

if witness_proofs:
    identity, proof = witness_proofs[0]
    print(f"Visualizing: {identity.expression}")
    print(f"  Status: {proof.status}  |  Nodes: {proof.node_count}")
    print(f"  Formula: {proof.lhs_formula}")
    fig = explorer.visualize_proof_interactive(proof)
    fig.show()
else:
    # Fall back: prove a hand-picked identity we know uses the witness tier
    print("No witness proofs from exploration — proving a catalog identity...")
    catalog_witness = [
        i for i in ALL_IDENTITIES
        if getattr(i, 'expected_method', None) == 'mcts'
    ]
    if catalog_witness:
        target = catalog_witness[0]
        proof = explorer.prove(target.expression)
        print(f"Proved: {proof.status}  |  Nodes: {proof.node_count}")
        if proof.witness_tree:
            fig = explorer.visualize_proof_interactive(proof)
            fig.show()
    else:
        print("(Try running more rounds or proving: exp(x) - exp(-x) == 2*sinh(x))")

## 7. Compress a proof — Occam's razor for EML witnesses

In [ ]:
if witness_proofs:
    identity, proof = witness_proofs[0]
    original_nodes = proof.node_count
    compressed = explorer.compress_proof(proof, n_simulations=500)
    print(f"Expression: {identity.expression}")
    print(f"  Before: {original_nodes} nodes")
    print(f"  After:  {compressed.node_count} nodes")
    if compressed.node_count < original_nodes:
        print(f"  Reduction: {original_nodes - compressed.node_count} nodes saved")
        print(f"  Compressed formula: {compressed.lhs_formula}")
    else:
        print("  Already minimal — no shorter witness found.")
else:
    print("No witness proofs available for compression demo.")

## 8. Transfer learning insight

Results from the 5×5 transfer matrix experiment (April 2026):

In [ ]:
import os

results_path = os.path.join(os.path.dirname(os.getcwd()), "python", "results", "transfer_learning.json")
# Try a few paths
for p in ["results/transfer_learning.json", "../results/transfer_learning.json"]:
    if os.path.exists(p):
        results_path = p
        break

if os.path.exists(results_path):
    with open(results_path) as f:
        transfer_data = json.load(f)

    matrix = transfer_data["matrix"]
    cats   = transfer_data["categories"]

    print("Transfer benefit matrix (>1.0 means training on row helps test on column):\n")
    header = f"{'':14s}" + "".join(f"{c:13s}" for c in cats)
    print(header)
    for train_cat in cats:
        row = f"{train_cat:14s}" + "".join(
            f"{matrix[train_cat][test_cat]:13.3f}" for test_cat in cats
        )
        print(row)

    print("\n→ Trig has the best transfer profile (1.05–1.32× to all domains).")
    print("→ This is why explore() seeds from trig by default.")
else:
    print("Transfer results not found — run: python -m monogate.frontiers.transfer_learning --full-matrix")

## 9. Learning curve from pre-run experiments

Batch pre-training produces no learning signal (scorer buffer stays at 0)
because 86% of catalog identities are solved before MCTS fires.
The `explore()` loop fixes this by generating hard-enough candidates.

In [ ]:
lc_path = "results/learning_curve.json"
if os.path.exists(lc_path):
    with open(lc_path) as f:
        lc_data = json.load(f)

    try:
        import matplotlib.pyplot as plt
        import numpy as np

        an_on  = lc_data["analysis_learning"]
        an_off = lc_data["analysis_no_learning"]

        n = len(an_on["mean_by_position"])
        x = range(n)
        m_on  = np.array(an_on["mean_by_position"])
        m_off = np.array(an_off["mean_by_position"])
        s_on  = np.array(an_on["std_by_position"])
        s_off = np.array(an_off["std_by_position"])

        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(x, m_on,  label=f"Learning ON  (ratio={an_on['improvement_ratio']:.2f}×)",  color="steelblue")
        ax.plot(x, m_off, label=f"Learning OFF (ratio={an_off['improvement_ratio']:.2f}×)", color="tomato")
        ax.fill_between(x, m_on  - s_on,  m_on  + s_on,  alpha=0.15, color="steelblue")
        ax.fill_between(x, m_off - s_off, m_off + s_off, alpha=0.15, color="tomato")
        ax.set_xlabel("Identity number (proof order)")
        ax.set_ylabel("Elapsed time (s)")
        ax.set_title("Neural Scorer Learning Curve — EMLProverV2 on 50 identities")
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

        summary = lc_data["summary"]
        print(f"Learning benefit: {summary['learning_benefit']:.2f}×")
        print(f"Prove rate (ON): {summary['prove_rate_learning']:.0%}  |  (OFF): {summary['prove_rate_no_learning']:.0%}")
    except ImportError:
        print("matplotlib not available")
        print(f"Learning benefit: {lc_data['summary']['learning_benefit']:.2f}×")
else:
    print("Run: python -m monogate.frontiers.learning_curve --n-identities 50 --n-runs 3")

## 10. Quick reference

In [ ]:
# The full API in one place
prover = EMLProverV2(enable_learning=True)

# Prove a single identity
r = prover.prove("exp(x) * exp(-x) == 1")
print(f"prove()  → {r.status} (conf={r.confidence:.2f}, {r.elapsed_s:.3f}s)")

# Generate conjectures
conjectures = prover.generate_conjectures(category="trig", n=10, temperature=0.7)
print(f"generate_conjectures() → {len(conjectures)} candidates")

# Run the exploration loop
res = prover.explore(n_rounds=2, n_per_round=10, verbose=False)
print(f"explore(2 rounds)      → {res['n_total_discovered']} discovered")

# Compress a proof (if witness available)
# compressed = prover.compress_proof(result)

# Visualize interactively (returns Plotly figure)
# fig = prover.visualize_proof_interactive(result)

# Visualize statically (matplotlib)
# prover.visualize_proof(result, style='tree', output_path='proof.png')

print("\nAll systems operational.")